# Exercise 02 — MovieLens bipartite network


## Setup

Import the packages needed for the exam.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from statistics import mean

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
pd.options.display.max_rows = 20

## Load the data

Read the MovieLens ratings and movie title files from the sample folder.

In [ ]:
ratings = pd.read_csv('../movielense/ml-latest-small/ratings.csv')
movies = pd.read_csv('../movielense/ml-latest-small/movies.csv')

print('ratings shape:', ratings.shape)
print('movies shape:', movies.shape)
ratings.head()

## Build a bipartite graph

Create a NetworkX graph where users and movies are separate node sets and ratings are edges between them.

In [ ]:
G = nx.Graph()
ratings = ratings.copy()
ratings['user_node'] = ratings['userId'].astype(str).radd('user_')
ratings['movie_node'] = ratings['movieId'].astype(str).radd('movie_')

user_nodes = ratings['user_node'].unique()
movie_nodes = ratings['movie_node'].unique()

G.add_nodes_from([(u, {'bipartite': 0, 'node_type': 'user'}) for u in user_nodes])
G.add_nodes_from([(m, {'bipartite': 1, 'node_type': 'movie'}) for m in movie_nodes])

for row in ratings.itertuples(index=False):
    G.add_edge(row.user_node, row.movie_node, rating=row.rating)

print('graph built with', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')

## Node and edge counts

Measure the size of the graph and its density.

In [ ]:
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)

num_users = sum(1 for _, d in G.nodes(data=True) if d['bipartite'] == 0)
num_movies = sum(1 for _, d in G.nodes(data=True) if d['bipartite'] == 1)

print(f'Total nodes: {num_nodes}')
print(f'Total edges: {num_edges}')
print(f'Users: {num_users}')
print(f'Movies: {num_movies}')
print(f'Density: {density:.6f}')

## Degree statistics

Compute degree summaries separately for users and movies.

In [ ]:
user_degrees = [(n, d) for n, d in G.degree() if G.nodes[n]['bipartite'] == 0]
movie_degrees = [(n, d) for n, d in G.degree() if G.nodes[n]['bipartite'] == 1]

user_deg_vals = [d for _, d in user_degrees]
movie_deg_vals = [d for _, d in movie_degrees]

print('User degree stats:')
print(' min', min(user_deg_vals), 'max', max(user_deg_vals), 'mean', f'{mean(user_deg_vals):.2f}')
print('Movie degree stats:')
print(' min', min(movie_deg_vals), 'max', max(movie_deg_vals), 'mean', f'{mean(movie_deg_vals):.2f}')

top_users = sorted(user_degrees, key=lambda x: x[1], reverse=True)[:5]
top_movies = sorted(movie_degrees, key=lambda x: x[1], reverse=True)[:5]
print('\nTop 5 users by degree:')
for n, d in top_users:
    print(n, d)
print('\nTop 5 movies by degree:')
for n, d in top_movies:
    print(n, d)

## Connectivity

Check the number of connected components and how large the main component is.

In [ ]:
components = list(nx.connected_components(G))
print('Connected components:', len(components))
print('Largest component size:', max(len(comp) for comp in components))

## Example path and cycle

Find one meaningful shortest path and one simple cycle inside the bipartite graph.

In [ ]:
movie_titles = movies.set_index('movieId')['title'].to_dict()

path_nodes = nx.shortest_path(G, source='movie_1', target='movie_3')
print('Shortest path from movie_1 to movie_3:')
for node in path_nodes:
    if node.startswith('movie_'):
        movie_id = int(node.replace('movie_', ''))
        print(' ', node, movie_titles.get(movie_id, 'unknown'))
    else:
        print(' ', node)
print('Path length:', len(path_nodes) - 1)

user_a = 'user_1'
user_b = 'user_2'
common_movies = set(G.neighbors(user_a)) & set(G.neighbors(user_b))
common_movies = sorted(common_movies)[:2]
cycle = [user_a, common_movies[0], user_b, common_movies[1], user_a] if len(common_movies) >= 2 else []
print('\nSample 4-cycle:')
for node in cycle:
    if node.startswith('movie_'):
        movie_id = int(node.replace('movie_', ''))
        print(' ', node, movie_titles.get(movie_id, 'unknown'))
    else:
        print(' ', node)

## Sample adjacency excerpt

Show a small subset of users and the movies they rated.

In [ ]:
small_users = [1, 2, 3, 4, 5]
small = ratings[ratings['userId'].isin(small_users)][['userId', 'movieId', 'rating']].merge(movies[['movieId', 'title']], on='movieId')
small = small.sort_values(['userId', 'movieId']).head(20)
small

## Representative visualization

Draw a small bipartite subgraph containing the most active users and most popular movies.

In [ ]:
top_users = [n for n, _ in sorted(user_degrees, key=lambda x: x[1], reverse=True)[:50]]
top_movies = [n for n, _ in sorted(movie_degrees, key=lambda x: x[1], reverse=True)[:50]]
sample_nodes = set(top_users + top_movies)
subG = G.subgraph(sample_nodes).copy()
pos = nx.bipartite_layout(subG, [n for n in subG if subG.nodes[n]['bipartite'] == 0])

plt.figure(figsize=(10, 7))
node_colors = ['skyblue' if subG.nodes[n]['bipartite'] == 0 else 'lightcoral' for n in subG]
labels = {n: n.replace('user_', 'U').replace('movie_', 'M') for n in subG}
nx.draw(subG, pos, labels=labels, node_color=node_colors, with_labels=True, node_size=900, font_size=8, edge_color='gray')
plt.title('Sample MovieLens bipartite subgraph: active users and popular movies')
plt.axis('off')
plt.show()